# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aijaz-khalique/flyrank-machine-learning-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content item for one client, summarized using the historical data available at the decision point.

Table: I use the content-level feature frame built from the FlyRank warehouse data.

Time window: I develop and verify the data contract on the mid-panel month 2026-03. I avoid the final month because it can overlap with the future outcome window used to define the label.

Prediction target: I predict is_declining_label, where 1 means the content is identified as declining and 0 means it is not declining.

Deliberately excluded: I exclude identifiers such as content_id and client_id from the model features because they identify entities rather than describing performance and could encourage memorization.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, clicks_90d, avg_position, content_age_days, and days_since_last_update.

Label: is_declining_label.

Context: baseline_rank, reason_codes, suggested_action_baseline, and trend_direction. These fields help describe or interpret the data but are not part of my five-feature model.

Excluded: content_id and client_id are excluded because they are identifiers. I also exclude the other baseline scores from the honest model because they are derived decision-support outputs rather than raw input signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
query_grain = """
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS row_count
FROM fact_content_daily_performance
WHERE month = '2026-03'
GROUP BY report_date, client_id, content_id
HAVING COUNT(*) > 1
LIMIT 10
"""

duckdb.sql(query_grain).df()




In [ ]:
query_window = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM fact_content_daily_performance
WHERE month = '2026-03'
"""

duckdb.sql(query_window).df()


In [ ]:

query_availability = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS available_rows
FROM fact_content_daily_performance
WHERE month = '2026-03'
"""

duckdb.sql(query_availability).df()

In [ ]:
features = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

label = "is_declining_label"

feature_frame = df[features + [label]].dropna().copy()

feature_frame.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = feature_frame[features]
y = feature_frame[label]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

honest_score = accuracy_score(y_test, predictions)

print("Honest accuracy:", round(honest_score, 4))

leaky_frame = feature_frame.copy()


leaky_frame["leaked_label"] = leaky_frame["is_declining_label"]

leaky_features = features + ["leaked_label"]

X_leak = leaky_frame[leaky_features]
y_leak = leaky_frame[label]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.2,
    random_state=42,
    stratify=y_leak
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_predictions = leaky_model.predict(X_test)

leaky_score = accuracy_score(y_test, leaky_predictions)

print("Leaky accuracy:", round(leaky_score, 4))

leaky_frame = leaky_frame.drop(columns=["leaked_label"])

print("Leaking feature removed.")
print("Honest accuracy:", round(honest_score, 4))


I deliberately added leaked_label, which was a direct copy of is_declining_label. The score jumped because the model could directly access the answer it was supposed to predict. This is target leakage, so I removed the column and kept the honest score from the model using only the five decision-time features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: This slice cannot prove why a content item is declining. It only contains observed search, traffic, and content signals. A relationship between a feature and the decline label should therefore be treated as directional and decision-support evidence, not as proof of causation. In addition, availability differences across the data history can limit the interpretation of engagement-related signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.